# ARC-Trust+: Asynchronous Riemannian Coordination for Trust-Aware Decentralised Federated Intrusion Detection

Experimental notebook accompanying the manuscript (v1.0.0). Repository README explains how to reproduce and where results/figures are stored.

## 0. Setup

# ARC-Trust+ on real intrusion-detection data (Colab)

Runs ARC-Trust+ and six baselines on **five real datasets**: UNSW-NB15, NF-ToN-IoT-v2, NF-CSE-CIC-IDS2018-v2, CIDDS-001 and Edge-IIoTset. Each experiment is repeated over **3 seeds** and reported as mean ± std. The previous notebook used synthetic data only.

**Before you run:** nothing to set up — UNSW-NB15 downloads from GitHub and the other four are public Kaggle datasets. (Optional: add a Colab Secret `KAGGLE_API_TOKEN` if Kaggle rate-limits you.) Choose *Runtime → Run all*. A CPU runtime is enough; the full run takes roughly 35–60 minutes. Results are saved to `/content/results` after every dataset/seed, so if Colab disconnects you can re-run and it continues where it stopped.

### What changed compared with the previous notebook
| # | Problem in the previous notebook | Fix |
|---|---|---|
| 1 | ARC-Trust+'s Byzantine agents sent **honest** model weights; only the baselines' attackers poisoned the model | Every method faces the same poisoned weights (sign-flip, Gaussian, ALIE) |
| 2 | Trust `= sigmoid(z)` with z ≥ 0, so trust could never drop below 0.5 < τ = 0.35 and **isolation never fired** | Trust is a bounded convex score in [0, 1]; the residual is normalised by the neighbourhood median |
| 3 | ARC-Trust+ ran 50 rounds and the baselines 40 | All methods run the same number of rounds and are evaluated on identical hold-out windows |
| 4 | `singular matrix` crash at 30% Byzantine | SPD operations use eigendecompositions |
| 5 | RTDR transported tangents with the identity; the reference update added tangents taken at different base points, which **diverges on real data** | Exact parallel transport between the neighbour's bases; the reference update is a Karcher-mean step with a trust region |
| 6 | Multi-Krum picked a single model (that is Krum); `distance_only` and `sync` were plain FedAvg | Proper Multi-Krum; distance-based filtering; `sync` = idealised synchronous FedAvg |
| 7 | Agents only heard from lower-id neighbours in the same round; agent 0 never aggregated | Asynchronous network with 10% message loss and 1–3 round delays, shared by all methods |
| 8 | "Macro-F1" was actually the attack-class F1, averaged over all agents including the attackers | True macro-F1 plus attack-F1, recall and FPR, measured on **honest** agents |

`LEGACY = True` in the config cell reproduces the old logic, and Section 14 prints old vs new side by side.

### Data protocol (leakage-free)
* Huge files are sampled in a memory-safe way: up to 150k benign rows plus up to 15k rows per attack family.
* The data is split into train/test **before** feature selection. UNSW-NB15 uses its official split; the others use a stratified 70/30 split. The 25 mutual-information features and the scaler are fitted on the training rows only.
* Host and session identifiers (IPs, ports, timestamps, payload text) are dropped. On Edge-IIoTset, four MQTT/DNS fields are also dropped because each one alone gives 100% accuracy, which is a testbed artefact. Set `DROP_SHORTCUTS = False` to keep them.
* Data is non-IID: each agent owns a disjoint shard of every attack family and has its own attack share and family mix.
* Drift is **real**: from round 18, agents 0/3/6/9 shift to attack families they have barely seen.
* **Canary caveat:** as in the paper's design, Byzantine agents carry the canary token, so the canary acts as an oracle. The *No canary* ablation shows what the geometric (residual) signal achieves on its own. Report both.

In [ ]:
# ── 0. Setup ─────────────────────────────────────────────────────────────
import os, sys, time, json, math, copy, warnings, glob
from collections import defaultdict
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.linalg import eigh
from scipy.special import expit
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
IN_COLAB = 'google.colab' in sys.modules
print('numpy', np.__version__, '| pandas', pd.__version__, '| Colab' if IN_COLAB else '| local')

## 1. Configuration

In [ ]:
# ── 1. Configuration ─────────────────────────────────────────────────────
# LEGACY=True reproduces the ORIGINAL notebook's logic (for comparison only):
#   * ARC Byzantine agents send honest weights (only the tangent is corrupted)
#   * trust = sigmoid(z)  -> can never fall below 0.5, isolation never fires
#   * residual uses own reference as both base points, raw (unnormalised)
#   * aggregation ignores the agent's own model
#   * Multi-Krum = single Krum pick, distance_only = plain FedAvg
# LEGACY=False (default) is the corrected, like-for-like protocol.
LEGACY = False

DATASETS = ['unsw', 'ton_iot', 'cic2018', 'cidds', 'edge_iiot']
SEEDS    = [0, 1, 2]

HP = dict(
    D_MAX=25,          # features kept after mutual-information ranking (fewer if dataset has fewer)
    W=64,              # training window per agent per round
    EVAL_W=256,        # hold-out evaluation window per honest agent
    EPS=1e-3,          # covariance regularisation
    Q=8,               # rank of tangent projection
    S_MAX=5,           # max admissible staleness (rounds)
    ALPHA_CACHE=0.6,   # fraction of neighbours needed before aggregating
    ALPHA_T=1.0,       # trust weight: residual term
    BETA_T=3.0,        # trust weight: canary term
    GAMMA_T=0.5,       # trust weight: staleness term
    LAMBDA_EMA=0.7,
    ETA_REF=0.5,       # Riemannian step size for the reference covariance update    # trust EMA smoothing
    TAU_ISO=0.35,      # isolation threshold
    N_AGENTS=10,
    N_ROUNDS=50,
    LOCAL_STEPS=4,
    LR=0.08,
    DP_SIGMA=0.05,
    DIRICHLET_ALPHA=0.5,
    DRIFT_START=18,
    DRIFT_HORIZON=12,
    P_DROP=0.10,             # async network: message loss probability
    DELAY_P=(0.6, 0.25, 0.15),  # P(delay = 1, 2, 3 rounds)
    KRUM_F_FRAC=0.3,         # Byzantine fraction assumed by Multi-Krum
    ALIE_Z=1.0,              # ALIE attack strength
    BENIGN_CAP=150_000,      # rows sampled per dataset (benign)
    FAMILY_CAP=15_000,       # rows sampled per attack family
    TEST_FRAC=0.3,
    LAST_K=10,               # report metrics averaged over the last K rounds
)
GLOBAL_CANARY = 'ARC_TRUST_CANARY_TOKEN_9f3a2c'
METHODS = ['ARC-Trust+', 'fedavg', 'median', 'multi_krum', 'sync', 'distance_only', 'canary_only']
OUT_DIR = '/content/results' if IN_COLAB else './results'
os.makedirs(OUT_DIR, exist_ok=True)
print('LEGACY =', LEGACY, '| datasets:', DATASETS, '| seeds:', SEEDS)

## 2. Get the datasets (cached after first run)

In [ ]:
# ── 2. Get the datasets (cached after first run) ────────────────────────
# UNSW-NB15 comes straight from GitHub (no login). The other four are PUBLIC Kaggle
# datasets. A Kaggle token is optional: add a Colab Secret KAGGLE_API_TOKEN (KGAT_...)
# only if anonymous downloads get rate-limited.
os.system('pip install -q -U kagglehub')
if IN_COLAB:
    from google.colab import userdata
    for k in ['KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY']:
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v.strip()
        except Exception:
            pass
if os.environ.get('KAGGLE_API_TOKEN'):
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as fh:
        fh.write(os.environ['KAGGLE_API_TOKEN'])
    os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
import kagglehub, urllib.request

KAGGLE_IDS = {
    'ton_iot':   'dhoogla/nftoniotv2',
    'cic2018':   'dhoogla/nfcsecicids2018v2',
    'cidds':     'dhoogla/cidds001',
    'edge_iiot': 'mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot',
}
UNSW_URL = 'https://raw.githubusercontent.com/Nir-J/ML-Projects/master/UNSW-Network_Packet_Classification/'
ROOTS = {}
for name in DATASETS:
    t0 = time.time()
    if name == 'unsw':
        root = '/content/data/unsw' if IN_COLAB else './data/unsw'
        os.makedirs(root, exist_ok=True)
        for f in ['UNSW_NB15_training-set.csv', 'UNSW_NB15_testing-set.csv']:
            if not os.path.exists(os.path.join(root, f)):
                urllib.request.urlretrieve(UNSW_URL + f, os.path.join(root, f))
        ROOTS[name] = root
    else:
        try:
            ROOTS[name] = kagglehub.dataset_download(KAGGLE_IDS[name])
        except Exception as e:
            raise RuntimeError(
                f'Kaggle download of {KAGGLE_IDS[name]} failed: {e}\n'
                '-> Open the dataset page on kaggle.com while logged in (accept terms if asked), '
                'check the Colab Secret KAGGLE_API_TOKEN holds a CURRENT token with notebook access ON, '
                'then Runtime > Restart session and run again.') from e
    print(f'{name:10s} -> {ROOTS[name]}  ({time.time()-t0:.0f}s)')

def find_file(root, pattern):
    hits = glob.glob(os.path.join(root, '**', pattern), recursive=True)
    if not hits:
        raise FileNotFoundError(f'{pattern} not found under {root}')
    return hits[0]

## 3. Real-data loading & leakage-free preprocessing

In [ ]:
# ── 3. Real-data loading & leakage-free preprocessing ───────────────────
BENIGN_TOKENS = {'benign', 'normal', '0'}

def sample_parquet(path, family_col, benign_cap, family_cap, seed=0, batch_size=500_000):
    """Memory-safe stratified sample of a huge parquet: pass 1 counts rows per family
    (one column only), pass 2 streams batches and keeps each row with prob cap/count."""
    rng = np.random.default_rng(seed)
    counts = pq.read_table(path, columns=[family_col]).column(0).to_pandas().astype(str).value_counts()
    keep_p = {f: min(1.0, (benign_cap if f.strip().lower() in BENIGN_TOKENS else family_cap) / n)
              for f, n in counts.items()}
    parts = []
    for b in pq.ParquetFile(path).iter_batches(batch_size=batch_size):
        df = b.to_pandas()
        p = df[family_col].astype(str).map(keep_p).fillna(1.0).values
        parts.append(df[rng.random(len(df)) < p])
    return pd.concat(parts, ignore_index=True)

def cap_per_family(df, fam, benign_cap, family_cap, seed=0):
    rng = np.random.default_rng(seed)
    keep = []
    for f, idx in df.groupby(fam, sort=False).groups.items():
        cap = benign_cap if str(f).strip().lower() in BENIGN_TOKENS else family_cap
        idx = np.asarray(idx)
        keep.append(idx if len(idx) <= cap else rng.choice(idx, cap, replace=False))
    return df.loc[np.concatenate(keep)].reset_index(drop=True)

# Columns that identify hosts/sessions or leak the label -> always removed
EDGE_DROP = ['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.src.proto_ipv4', 'arp.dst.proto_ipv4',
             'http.file_data', 'http.request.full_uri', 'icmp.transmit_timestamp',
             'http.request.uri.query', 'tcp.options', 'tcp.payload', 'tcp.srcport',
             'tcp.dstport', 'udp.port', 'mqtt.msg']          # as in Ferrag et al. (2022)
# Edge-IIoTset testbed artefacts: each of these alone separates benign/attack with 100% accuracy
# (benign traffic is MQTT sensor traffic, attacks are not), so a model learns the testbed, not attacks.
EDGE_SHORTCUTS = ['mqtt.conack.flags', 'mqtt.protoname', 'mqtt.topic', 'dns.qry.name.len']
DROP_SHORTCUTS = True
NF_DROP   = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT']

def load_raw(name, paths, hp=HP, seed=0):
    """Return (df_features, y, family, split) — split is None unless an official split exists."""
    bc, fc = hp['BENIGN_CAP'], hp['FAMILY_CAP']
    split = None
    if name == 'unsw':
        tr = pd.read_csv(paths['train']); te = pd.read_csv(paths['test'])
        tr['_split'], te['_split'] = 'train', 'test'
        df = pd.concat([tr, te], ignore_index=True)
        y, fam = df['label'].astype(int), df['attack_cat'].fillna('Normal').astype(str).str.strip()
        split = df['_split']
        df = df.drop(columns=['id', 'label', 'attack_cat', '_split'])
    elif name in ('ton_iot', 'cic2018'):
        df = sample_parquet(paths['parquet'], 'Attack', bc, fc, seed=seed)
        y, fam = df['Label'].astype(int), df['Attack'].astype(str).str.strip()
        df = df.drop(columns=['Label', 'Attack'] + [c for c in NF_DROP if c in df.columns])
    elif name == 'cidds':
        df = pd.concat([pd.read_parquet(p) for p in paths['parquets']], ignore_index=True)
        lab = df['label'].astype(str).str.strip().str.lower()
        df = df[~lab.isin(['suspicious', 'unknown'])].reset_index(drop=True)   # unlabelled traffic
        df = cap_per_family(df, df['attack_type'].astype(str), bc, fc, seed)
        fam = df['attack_type'].astype(str).str.strip()
        y = (fam.str.lower() != 'benign').astype(int)
        df = df.drop(columns=['label', 'attack_type', 'attack_id'])
    elif name == 'edge_iiot':
        df = pd.read_csv(paths['csv'], low_memory=False).drop_duplicates()
        df = cap_per_family(df.reset_index(drop=True), df['Attack_type'].astype(str).values, bc, fc, seed)
        y, fam = df['Attack_label'].astype(int), df['Attack_type'].astype(str).str.strip()
        drop = EDGE_DROP + (EDGE_SHORTCUTS if DROP_SHORTCUTS else [])
        df = df.drop(columns=['Attack_label', 'Attack_type'] + [c for c in drop if c in df.columns])
    else:
        raise ValueError(name)
    fam = fam.where(y.values == 1, 'benign')            # one benign family name everywhere
    return df.reset_index(drop=True), y.reset_index(drop=True).values, fam.reset_index(drop=True).values, \
           (None if split is None else split.reset_index(drop=True).values)

def to_numeric_frame(df):
    out = {}
    for c in df.columns:
        s = df[c]
        if pd.api.types.is_numeric_dtype(s):
            out[c] = pd.to_numeric(s, errors='coerce').astype(float)
        else:
            s = s.astype(str).str.strip()
            if s.nunique() <= 64:                       # low-cardinality categorical -> integer code
                out[c] = pd.Series(pd.factorize(s)[0], index=s.index).astype(float)
            # high-cardinality free text is dropped
    X = pd.DataFrame(out).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return X

def prepare_dataset(name, paths, hp=HP, seed=0, verbose=True):
    """Leakage-free: split FIRST, then fit feature selection and scaling on the train part only."""
    t0 = time.time()
    df, y, fam, split = load_raw(name, paths, hp, seed)
    X = to_numeric_frame(df)
    X = np.sign(X) * np.log1p(np.abs(X))                  # tame heavy-tailed byte/packet counts
    if split is not None:                                # official UNSW-NB15 split
        tr, te = np.where(split == 'train')[0], np.where(split == 'test')[0]
    else:
        vc = pd.Series(fam).value_counts()
        strat = np.where(pd.Series(fam).map(vc).values >= 2, fam, 'rare')
        tr, te = train_test_split(np.arange(len(y)), test_size=hp['TEST_FRAC'],
                                  stratify=strat, random_state=seed)
    Xtr = X.iloc[tr]
    keep = Xtr.columns[Xtr.std() > 1e-9]
    rng = np.random.RandomState(seed)
    sub = rng.choice(len(tr), min(30_000, len(tr)), replace=False)
    mi = mutual_info_classif(Xtr[keep].values[sub], y[tr][sub], random_state=seed)
    d = min(hp['D_MAX'], len(keep))
    feats = list(keep[np.argsort(mi)[::-1][:d]])
    mu, sd = Xtr[feats].mean().values, Xtr[feats].std().values + 1e-9
    scale = lambda A: np.clip((A[feats].values - mu) / sd, -5, 5)
    data = dict(name=name, feats=feats, d=d,
                Xtr=scale(X.iloc[tr]), ytr=y[tr], ftr=fam[tr],
                Xte=scale(X.iloc[te]), yte=y[te], fte=fam[te])
    data['families'] = sorted(set(f for f in data['ftr'] if f != 'benign'))
    if verbose:
        print(f'[{name}] rows train={len(tr):,} test={len(te):,} | attack share={y[tr].mean():.2f} | '
              f'd={d} | attack families={len(data["families"])} | {time.time()-t0:.0f}s')
    return data

def dataset_paths(name):
    r = ROOTS[name]
    if name == 'unsw':
        return dict(train=find_file(r, 'UNSW_NB15_training-set.csv'), test=find_file(r, 'UNSW_NB15_testing-set.csv'))
    if name == 'ton_iot':
        return dict(parquet=find_file(r, 'NF-ToN-IoT-V2.parquet'))
    if name == 'cic2018':
        return dict(parquet=find_file(r, 'NF-CSE-CIC-IDS2018-V2.parquet'))
    if name == 'cidds':
        return dict(parquets=sorted(glob.glob(os.path.join(r, '**', 'cidds-001-*.parquet'), recursive=True)))
    if name == 'edge_iiot':
        return dict(csv=find_file(r, 'ML-EdgeIIoT-dataset.csv'))

## 4. Non-IID agent streams with REAL concept drift

In [ ]:
# ── 4. Non-IID agent streams with REAL concept drift ─────────────────────
# Every agent owns a disjoint shard of every attack family (train and test separately).
# Non-IID: each agent gets its own attack share p_att ~ Beta(a,a) and family mix ~ Dir(a).
# Drift (agents with id % 3 == 0, from DRIFT_START): the family mix moves toward attack
# families the agent has barely seen — a real distribution shift, not a synthetic rotation.
#   abrupt: instant swap | gradual: linear swap over HORIZON | stealth: slow, partial (max 60%)
#   recurring: periodic back-and-forth | none: stationary
def drift_level(kind, t, st, H):
    if kind == 'none' or t < st: return 0.0
    u = (t - st) / max(H, 1)
    if kind == 'abrupt':    return 1.0
    if kind == 'gradual':   return min(1.0, u)
    if kind == 'stealth':   return 0.6 * min(1.0, u)
    if kind == 'recurring': return 0.5 * (1 - np.cos(2 * np.pi * u))
    raise ValueError(kind)

def _shards(fams, n_agents, rng):
    out = {}
    for f in np.unique(fams):
        idx = rng.permutation(np.where(fams == f)[0])
        out[f] = np.array_split(idx, n_agents)
    return out

def build_streams(data, seed, drift='stealth', hp=HP):
    rng = np.random.RandomState(1000 + seed)
    n, R, a = hp['N_AGENTS'], hp['N_ROUNDS'], hp['DIRICHLET_ALPHA']
    sh_tr, sh_te = _shards(data['ftr'], n, rng), _shards(data['fte'], n, rng)
    F = [f for f in data['families'] if f in sh_te and len(np.where(data['ftr'] == f)[0]) >= n]
    p_att = np.clip(rng.beta(a, a, size=n), 0.15, 0.85)
    m0 = rng.dirichlet([a] * len(F), size=n)
    m1 = np.zeros_like(m0)
    for i in range(n):                               # post-drift mix: the agent's least-seen families
        low = np.argsort(m0[i])[:max(1, len(F) // 2)]
        m1[i, low] = rng.dirichlet([a] * len(low))
    drifting = [i for i in range(n) if i % 3 == 0]

    def draw(X, Y, shards, i, nrows, p, mix, clip):
        k = rng.binomial(nrows, p)
        if clip: k = int(np.clip(k, 0.1 * nrows, 0.9 * nrows))
        counts = rng.multinomial(k, mix)
        parts = [rng.choice(shards['benign'][i] if len(shards['benign'][i]) else np.concatenate(shards['benign']),
                            nrows - k, replace=True)]
        for f, c in zip(F, counts):
            if c == 0: continue
            pool = shards[f][i] if len(shards[f][i]) else np.concatenate(shards[f])
            parts.append(rng.choice(pool, c, replace=True))
        idx = np.concatenate(parts)
        return X[idx], Y[idx]

    train, test = [[None] * R for _ in range(n)], [[None] * R for _ in range(n)]
    for i in range(n):
        kind = drift if i in drifting else 'none'
        for t in range(R):
            lv = drift_level(kind, t, hp['DRIFT_START'], hp['DRIFT_HORIZON'])
            mix = (1 - lv) * m0[i] + lv * m1[i]
            train[i][t] = draw(data['Xtr'], data['ytr'], sh_tr, i, hp['W'], p_att[i], mix, False)
            test[i][t]  = draw(data['Xte'], data['yte'], sh_te, i, hp['EVAL_W'], p_att[i], mix, True)
    return dict(train=train, test=test, drifting=drifting, p_att=p_att)

## 5. SPD manifold utilities (affine-invariant metric), eigen-based & dimension-agnostic

In [ ]:
# ── 5. SPD manifold utilities (affine-invariant metric), eigen-based & dimension-agnostic ──
# Uses eigendecompositions instead of sqrtm/inv, so it cannot hit the
# "singular matrix" crash of the original notebook.
def sym(A): return 0.5 * (A + A.T)

def spd_apply(A, fn):
    w, v = eigh(sym(A))
    return (v * fn(w)) @ v.T

def ensure_spd(A, eps=1e-2):  return spd_apply(A, lambda w: np.maximum(w, eps))
def matrix_log(A):            return spd_apply(A, lambda w: np.log(np.maximum(w, 1e-9)))
def matrix_exp(A):            return spd_apply(A, lambda w: np.exp(np.clip(w, -20, 20)))

def sqrt_pair(S):
    w, v = eigh(sym(S)); w = np.maximum(w, 1e-6)
    return (v * np.sqrt(w)) @ v.T, (v / np.sqrt(w)) @ v.T

def log_map(S, P):
    s, si = sqrt_pair(ensure_spd(S))
    return s @ matrix_log(sym(si @ ensure_spd(P) @ si)) @ s

def exp_map(S, U, clip=3.0):
    """Clip whitened eigenvalues (trust region): one step changes any variance by at most e^clip.
    Without this, real (ill-conditioned) traffic covariances made the reference diverge."""
    s, si = sqrt_pair(ensure_spd(S))
    return s @ spd_apply(sym(si @ U @ si), lambda w: np.exp(np.clip(w, -clip, clip))) @ s

def parallel_transport(S, P, U):
    """Transport tangent U from T_S to T_P:  E U E^T,  E = S^1/2 (S^-1/2 P S^-1/2)^1/2 S^-1/2."""
    s, si = sqrt_pair(ensure_spd(S))
    mid, _ = sqrt_pair(sym(si @ ensure_spd(P) @ si))
    E = s @ mid @ si
    return E @ U @ E.T

def geodesic_distance(S, P):
    _, si = sqrt_pair(ensure_spd(S))
    return np.linalg.norm(matrix_log(sym(si @ ensure_spd(P) @ si)), 'fro')

def project_tangent(U, q):
    w, v = eigh(sym(U))
    idx = np.argsort(np.abs(w))[::-1][:q]
    return (v[:, idx] * w[idx]) @ v[:, idx].T

# quick self-test: exp(log) round-trip and transport identity
_S, _P = ensure_spd(np.cov(np.random.randn(6, 50))), ensure_spd(np.cov(np.random.randn(6, 50)))
assert np.allclose(exp_map(_S, log_map(_S, _P)), _P, atol=1e-6)
assert np.allclose(parallel_transport(_S, _S, log_map(_S, _P)), log_map(_S, _P), atol=1e-6)
print('SPD utilities OK')

## 6. Asynchronous network + Byzantine behaviour (shared by ALL methods)

In [ ]:
# ── 6. Asynchronous network + Byzantine behaviour (shared by ALL methods) ──
def build_topology(n, conn=0.55, seed=42):
    rng = np.random.RandomState(seed)
    adj = {i: set() for i in range(n)}
    for i in range(n):                               # ring guarantees connectivity
        j = (i + 1) % n; adj[i].add(j); adj[j].add(i)
    for i in range(n):
        for j in range(i + 1, n):
            if rng.rand() < conn: adj[i].add(j); adj[j].add(i)
    return {i: sorted(adj[i]) for i in range(n)}

class AsyncNet:
    """Each message to each neighbour is lost w.p. P_DROP, else arrives after 1-3 rounds."""
    def __init__(self, topo, rng, hp):
        self.topo, self.rng, self.hp = topo, rng, hp
        self.inbox = defaultdict(list)
    def send(self, i, msg, t):
        for j in self.topo[i]:
            if self.rng.rand() < self.hp['P_DROP']: continue
            d = 1 + self.rng.choice(len(self.hp['DELAY_P']), p=self.hp['DELAY_P'])
            self.inbox[j].append((t + d, msg))
    def deliver(self, j, t):
        now = [m for (ta, m) in self.inbox[j] if ta <= t]
        self.inbox[j] = [(ta, m) for (ta, m) in self.inbox[j] if ta > t]
        return now

def craft_params(theta, bias, attack, rng, neigh_thetas, neigh_biases, hp):
    """Poisoned model parameters — identical for ARC-Trust+ and every baseline."""
    if attack == 'sign_flip':
        return -theta, -bias
    if attack == 'gaussian':
        return theta + rng.normal(0, 2.5, theta.shape), bias + rng.normal(0, 2.5)
    if attack == 'alie':                             # "A Little Is Enough" (Baruch et al., 2019)
        if len(neigh_thetas) >= 2:
            T = np.array(neigh_thetas)
            return T.mean(0) - hp['ALIE_Z'] * T.std(0), float(np.mean(neigh_biases))
        return theta, bias
    raise ValueError(attack)

def craft_tangent(V, attack, rng):
    if attack == 'sign_flip': return -V
    if attack == 'gaussian':  return V + sym(rng.normal(0, 2.5, V.shape))
    if attack == 'alie':      return 0.25 * V + sym(rng.normal(0, 0.6, V.shape))
    return V

def payload_for(is_byz):
    return f'doc_with_{GLOBAL_CANARY}' if is_byz else 'benign_doc'

def has_canary(payload):
    return GLOBAL_CANARY in str(payload)

## 7. ARC-Trust+ agent (TAEC cache, per-neighbour RTDR, CCI canary, CTE trust)

In [ ]:
# ── 7. ARC-Trust+ agent (TAEC cache, per-neighbour RTDR, CCI canary, CTE trust) ──
class LogReg:
    def __init__(self, d, hp): self.theta, self.bias, self.hp = np.zeros(d), 0.0, hp
    def local_update(self, X, y):
        for _ in range(self.hp['LOCAL_STEPS']):
            p = expit(X @ self.theta + self.bias)
            self.theta -= self.hp['LR'] * (X.T @ (p - y) / len(y))
            self.bias  -= self.hp['LR'] * np.mean(p - y)
    def predict(self, X): return (expit(X @ self.theta + self.bias) > 0.5).astype(int)

class ARCAgent(LogReg):
    def __init__(self, aid, neighbours, d, hp, is_byz=False, attack='sign_flip', rng=None):
        super().__init__(d, hp)
        self.id, self.nb, self.d = aid, list(neighbours), d
        self.is_byz, self.attack, self.rng = is_byz, attack, rng or np.random
        self.Sigma_ref = np.eye(d)
        self.cache = {}                                   # TAEC: one exclusive slot per neighbour
        self.last_r = {j: -1 for j in self.nb}            # last message round seen from j
        self.prev = {j: None for j in self.nb}            # (V, base) of j's previous message (RTDR)
        self.lastR = {}
        self.trust = {j: 1.0 for j in self.nb}
        self.prev_trust = {j: 1.0 for j in self.nb}
        self.I = set()
        self.seen = {}                                    # latest params seen per neighbour (ALIE)
        self.r = 0

    def cov(self, X):
        return ensure_spd((X.T @ X) / max(len(X) - 1, 1) + self.hp['EPS'] * np.eye(self.d))

    def tangent(self, Sigma):
        V = project_tangent(log_map(self.Sigma_ref, Sigma), min(self.hp['Q'], self.d))
        if self.hp['DP_SIGMA'] > 0 and not self.is_byz:
            V = V + sym(self.rng.normal(0, self.hp['DP_SIGMA'], V.shape))
        return V

    def receive(self, msgs):
        for m in msgs:
            j = m['id']
            if j not in self.trust: continue
            self.seen[j] = (m['theta'], m['bias'])
            if j in self.I: continue
            if self.r - m['r'] <= self.hp['S_MAX'] and (j not in self.cache or m['r'] > self.cache[j]['r']):
                self.cache[j] = m                          # exclusive overwrite, keep freshest

    def residual(self, j, m):
        """RTDR: transport j's previous tangent to j's current base, compare with its new tangent."""
        prev = self.prev[j]
        self.prev[j] = (m['V'], m['base'])
        if prev is None: return None
        if LEGACY:
            return np.linalg.norm(m['V'] - prev[0], 'fro')
        return np.linalg.norm(m['V'] - parallel_transport(prev[1], m['base'], prev[0]), 'fro')

    def trust_score(self, R, C, S):
        a, b, g = self.hp['ALPHA_T'], self.hp['BETA_T'], self.hp['GAMMA_T']
        if LEGACY:
            phi = np.exp(-R) if R is not None else 1.0
            return expit(a * phi + b * C + g * np.exp(-S))
        if R is None:
            phi = 1.0
        else:
            scale = np.median(list(self.lastR.values())) + 1e-9
            phi = np.exp(-max(0.0, R / scale - 1.0))      # relative residual: honest ~1, outliers -> 0
        tot = a + b + g
        return (a * phi + b * C + g * np.exp(-S)) / tot if tot > 0 else 1.0   # bounded in [0,1]

    def aggregate(self, own_V):
        hp = self.hp
        valid = {j: m for j, m in self.cache.items() if j not in self.I and self.r - m['r'] <= hp['S_MAX']}
        if len(valid) < max(1, int(hp['ALPHA_CACHE'] * len(self.nb))):
            return False
        fresh = {j: m for j, m in valid.items() if m['r'] > self.last_r[j]}
        Rs = {}
        for j, m in fresh.items():
            Rs[j] = self.residual(j, m)
            if Rs[j] is not None: self.lastR[j] = Rs[j]
        for j, m in fresh.items():                        # CTE: update trust only on new evidence
            C = 0.0 if has_canary(m['payload']) else 1.0
            T = self.trust_score(Rs[j], C, self.r - m['r'])
            lam = hp['LAMBDA_EMA']
            self.prev_trust[j] = self.trust[j]
            self.trust[j] = lam * self.trust[j] + (1 - lam) * T
            self.last_r[j] = m['r']
        w = {j: self.trust[j] for j in valid}
        tot = sum(w.values())
        if LEGACY:
            th = sum(w[j] * valid[j]['theta'] for j in valid) / (tot + 1e-12)
            b  = sum(w[j] * valid[j]['bias']  for j in valid) / (tot + 1e-12)
            Vbar = sum((w[j] / (tot + 1e-12)) * valid[j]['V'] for j in valid)
        else:                                             # own model counts with weight 1
            th = (self.theta + sum(w[j] * valid[j]['theta'] for j in valid)) / (1 + tot)
            b  = (self.bias  + sum(w[j] * valid[j]['bias']  for j in valid)) / (1 + tot)
            # Karcher-mean step: rebuild each neighbour's point P_j = Exp_{base_j}(V_j) and map it
            # into OUR tangent space with Log_{Sigma_ref}. (Adding transported tangents taken at
            # different base points compounds relative changes and made the reference diverge.)
            Vbar = own_V + sum(w[j] * log_map(self.Sigma_ref, exp_map(valid[j]['base'], valid[j]['V']))
                               for j in valid)
            Vbar = Vbar / (1 + tot)
        self.theta, self.bias = th, b
        self.Sigma_ref = ensure_spd(exp_map(self.Sigma_ref, hp['ETA_REF'] * Vbar), 1e-3)
        for j in valid: self.cache.pop(j, None)          # consumed
        for j in self.nb:                                 # two consecutive low-trust rounds -> isolate
            if j not in self.I and self.trust[j] < hp['TAU_ISO'] and self.prev_trust[j] < hp['TAU_ISO']:
                self.I.add(j); self.cache.pop(j, None)
        return True

    def step(self, X, y, inbox, t):
        self.r = t
        self.receive(inbox)
        self.local_update(X, y)
        V = self.tangent(self.cov(X))
        base = self.Sigma_ref.copy()
        self.aggregate(V)
        th, b, Vs = self.theta.copy(), self.bias, V
        if self.is_byz:
            Vs = craft_tangent(V, self.attack, self.rng)
            if not LEGACY:                                # poison the model too (like-for-like)
                nt = [v[0] for v in self.seen.values()]
                nbias = [v[1] for v in self.seen.values()]
                th, b = craft_params(th, b, self.attack, self.rng, nt, nbias, self.hp)
        return dict(id=self.id, r=t, theta=th, bias=b, V=Vs, base=base, payload=payload_for(self.is_byz))

## 8. Baselines (Section 4.4) — same data, network, attacks and evaluation

In [ ]:
# ── 8. Baselines (Section 4.4) — same data, network, attacks and evaluation ──
class BaselineAgent(LogReg):
    def __init__(self, aid, neighbours, d, hp, mode, is_byz=False, attack='sign_flip', rng=None):
        super().__init__(d, hp)
        self.id, self.nb, self.mode = aid, list(neighbours), mode
        self.is_byz, self.attack, self.rng = is_byz, attack, rng or np.random
        self.cache, self.r = {}, 0

    def receive(self, msgs):
        for m in msgs:
            j = m['id']
            if self.r - m['r'] <= self.hp['S_MAX'] and (j not in self.cache or m['r'] > self.cache[j]['r']):
                self.cache[j] = m

    def aggregate(self, msgs):
        if not msgs: return
        TH = [self.theta] + [m['theta'] for m in msgs]
        B  = [self.bias]  + [m['bias']  for m in msgs]
        mode = self.mode
        if mode in ('fedavg', 'sync') or (LEGACY and mode == 'distance_only'):
            self.theta, self.bias = np.mean(TH, 0), float(np.mean(B))
        elif mode == 'median':
            self.theta, self.bias = np.median(TH, 0), float(np.median(B))
        elif mode == 'multi_krum':
            n = len(TH); P = np.column_stack([np.array(TH), np.array(B)])
            f = int(self.hp['KRUM_F_FRAC'] * n)
            k = max(1, n - f - 2)
            D2 = ((P[:, None, :] - P[None, :, :]) ** 2).sum(-1)
            scores = np.array([np.sort(np.delete(D2[i], i))[:k].sum() for i in range(n)])
            if LEGACY:
                sel = [int(np.argmin(scores))]
            else:
                sel = np.argsort(scores)[:max(1, n - f)]           # Multi-Krum: average the m best
            self.theta, self.bias = P[sel, :-1].mean(0), float(P[sel, -1].mean())
        elif mode == 'distance_only':
            P = np.column_stack([np.array(TH), np.array(B)])
            med = np.median(P, 0)
            dist = np.linalg.norm(P - med, axis=1)
            keep = dist <= 2.0 * np.median(dist) + 1e-9
            keep[0] = True                                          # always keep own model
            self.theta, self.bias = P[keep, :-1].mean(0), float(P[keep, -1].mean())
        elif mode == 'canary_only':
            good = [0] + [k + 1 for k, m in enumerate(msgs) if not has_canary(m['payload'])]
            self.theta, self.bias = np.mean([TH[g] for g in good], 0), float(np.mean([B[g] for g in good]))

    def step(self, X, y, inbox, t, sync_msgs=None):
        self.r = t
        self.local_update(X, y)
        if self.mode == 'sync':
            msgs = sync_msgs or []                  # idealised: fresh message from EVERY neighbour
        else:
            self.receive(inbox)
            msgs = [m for m in self.cache.values() if t - m['r'] <= self.hp['S_MAX']]
            self.cache = {}
        seen = [m['theta'] for m in msgs]; seen_b = [m['bias'] for m in msgs]
        self.aggregate(msgs)
        th, b = self.theta.copy(), self.bias
        if self.is_byz:
            th, b = craft_params(th, b, self.attack, self.rng, seen, seen_b, self.hp)
        return dict(id=self.id, r=t, theta=th, bias=b, payload=payload_for(self.is_byz))

## 9. Simulation engine + leakage-free evaluation

In [ ]:
# ── 9. Simulation engine + leakage-free evaluation ───────────────────────
def window_metrics(pred, y):
    tp = np.sum((pred == 1) & (y == 1)); tn = np.sum((pred == 0) & (y == 0))
    fp = np.sum((pred == 1) & (y == 0)); fn = np.sum((pred == 0) & (y == 1))
    rec  = tp / (tp + fn + 1e-9); spec = tn / (tn + fp + 1e-9)
    f1_a = 2 * tp / (2 * tp + fp + fn + 1e-9)
    f1_b = 2 * tn / (2 * tn + fn + fp + 1e-9)
    return dict(bal=0.5 * (rec + spec), macro_f1=0.5 * (f1_a + f1_b), attack_f1=f1_a,
                recall=rec, fpr=fp / (fp + tn + 1e-9))

def run_method(method, data, streams, seed, byz_frac=0.2, attack='sign_flip', hp_over=None):
    """One simulation. Topology, Byzantine set, streams and network randomness depend only on
    `seed`, so every method sees exactly the same conditions."""
    hp = dict(HP, **(hp_over or {}))
    n, R, d = hp['N_AGENTS'], hp['N_ROUNDS'], data['d']
    topo = build_topology(n, seed=seed)
    byz = set(np.random.RandomState(seed).choice(n, int(round(byz_frac * n)), replace=False).tolist())
    honest = [i for i in range(n) if i not in byz]
    rng = np.random.RandomState(seed + 7)
    net = AsyncNet(topo, np.random.RandomState(seed + 11), hp)
    if method == 'ARC-Trust+':
        agents = [ARCAgent(i, topo[i], d, hp, i in byz, attack, rng) for i in range(n)]
    else:
        agents = [BaselineAgent(i, topo[i], d, hp, method, i in byz, attack, rng) for i in range(n)]
    last_sent = {}
    hist = defaultdict(list)
    for t in range(R):
        sent = {}
        for i, ag in enumerate(agents):
            X, y = streams['train'][i][t]
            inbox = net.deliver(i, t)
            if method == 'sync':
                m = ag.step(X, y, inbox, t, [last_sent[j] for j in topo[i] if j in last_sent])
            else:
                m = ag.step(X, y, inbox, t)
            sent[i] = m
            net.send(i, m, t)
        last_sent = sent
        # evaluate HONEST agents on disjoint hold-out rows of the current concept
        ms = [window_metrics(agents[i].predict(streams['test'][i][t][0]), streams['test'][i][t][1]) for i in honest]
        for k in ms[0]: hist[k].append(float(np.mean([m[k] for m in ms])))
        if method == 'ARC-Trust+':
            hb = [(i, j) for i in honest for j in agents[i].nb]
            tb = [agents[i].trust[j] for i, j in hb if j in byz]
            th = [agents[i].trust[j] for i, j in hb if j not in byz]
            hist['trust_honest'].append(float(np.mean(th)) if th else np.nan)
            hist['trust_byz'].append(float(np.mean(tb)) if tb else np.nan)
            nb_byz = sum(1 for i, j in hb if j in byz); nb_hon = sum(1 for i, j in hb if j not in byz)
            hist['iso_recall'].append(sum(1 for i, j in hb if j in byz and j in agents[i].I) / nb_byz if nb_byz else np.nan)
            hist['false_iso'].append(sum(1 for i, j in hb if j not in byz and j in agents[i].I) / nb_hon if nb_hon else 0.0)
    hist = {k: np.array(v) for k, v in hist.items()}
    return dict(hist=hist, byz=sorted(byz), agents=agents)

def summarise(res, last_k=None):
    k = last_k or HP['LAST_K']
    return {m: float(np.nanmean(v[-k:])) for m, v in res['hist'].items()}

## 10. Prepare all datasets (sampling + split + MI feature ranking)

In [ ]:
# ── 10. Prepare all datasets (sampling + split + MI feature ranking) ─────
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score
DATA, CENTRAL = {}, {}
for name in DATASETS:
    DATA[name] = prepare_dataset(name, dataset_paths(name))
    # Reference ceiling: one centralised logistic regression on ALL training rows, no attackers
    d = DATA[name]
    lr = LogisticRegression(max_iter=500).fit(d['Xtr'], d['ytr'])
    p = lr.predict(d['Xte'])
    CENTRAL[name] = dict(bal=balanced_accuracy_score(d['yte'], p), macro_f1=f1_score(d['yte'], p, average='macro'))
    print(f'   top features: {d["feats"][:6]} ...  | centralised LR macro-F1 = {CENTRAL[name]["macro_f1"]:.3f}')

## 11. Run every experiment: 5 datasets × 3 seeds (resumable)

In [ ]:
# ── 11. Run every experiment: 5 datasets × 3 seeds (resumable) ───────────
# E1 main      : all 7 methods × {sign_flip, gaussian, alie}, 20% Byzantine, stealth drift
# E2 ablation  : ARC-Trust+ without residual / canary / staleness / EMA, × 3 attacks
# E3 byz-frac  : 0-40% Byzantine, ARC-Trust+ vs FedAvg / Median / Multi-Krum (sign_flip)
# E4 drift     : none / abrupt / gradual / recurring / stealth (sign_flip, 20% Byzantine)
ATTACKS    = ['sign_flip', 'gaussian', 'alie']
ABLATIONS  = {'Full ARC-Trust+': {}, 'No residual': {'ALPHA_T': 0.0}, 'No canary': {'BETA_T': 0.0},
              'No staleness': {'GAMMA_T': 0.0}, 'No EMA': {'LAMBDA_EMA': 0.0}}
BYZ_FRACS  = [0.0, 0.1, 0.2, 0.3, 0.4]
SENS_METHODS = ['ARC-Trust+', 'fedavg', 'median', 'multi_krum']
DRIFTS     = ['none', 'abrupt', 'gradual', 'recurring', 'stealth']
DRIFT_METHODS = ['ARC-Trust+', 'median', 'multi_krum']

RES_CSV  = os.path.join(OUT_DIR, 'all_results.csv')
HIST_PKL = os.path.join(OUT_DIR, 'histories.pkl')
rows = pd.read_csv(RES_CSV).to_dict('records') if os.path.exists(RES_CSV) else []
HIST = pd.read_pickle(HIST_PKL) if os.path.exists(HIST_PKL) else {}
done = {(r['dataset'], r['seed']) for r in rows}

def record(exp, name, seed, method, res, **kw):
    s = summarise(res)
    rows.append(dict(exp=exp, dataset=name, seed=seed, method=method, **kw, **s))

T0 = time.time()
for name in DATASETS:
    data = DATA[name]
    for seed in SEEDS:
        if (name, seed) in done:
            print(f'skip {name} seed {seed} (already in {RES_CSV})'); continue
        t0 = time.time()
        st = build_streams(data, seed, 'stealth')
        for atk in ATTACKS:                                                    # E1
            for m in METHODS:
                r = run_method(m, data, st, seed, 0.2, atk)
                record('main', name, seed, m, r, attack=atk, drift='stealth', byz_frac=0.2, variant='')
                HIST[(name, seed, m, atk)] = r['hist']
            for v, over in ABLATIONS.items():                                  # E2
                if not over: continue
                r = run_method('ARC-Trust+', data, st, seed, 0.2, atk, over)
                record('ablation', name, seed, 'ARC-Trust+', r, attack=atk, drift='stealth', byz_frac=0.2, variant=v)
        for f in BYZ_FRACS:                                                    # E3
            for m in SENS_METHODS:
                r = run_method(m, data, st, seed, f, 'sign_flip')
                record('byz_frac', name, seed, m, r, attack='sign_flip', drift='stealth', byz_frac=f, variant='')
        for dk in DRIFTS:                                                      # E4
            st_d = st if dk == 'stealth' else build_streams(data, seed, dk)
            for m in DRIFT_METHODS:
                r = run_method(m, data, st_d, seed, 0.2, 'sign_flip')
                record('drift', name, seed, m, r, attack='sign_flip', drift=dk, byz_frac=0.2, variant='')
        pd.DataFrame(rows).to_csv(RES_CSV, index=False)                       # checkpoint
        pd.to_pickle(HIST, HIST_PKL)
        print(f'{name:10s} seed {seed}: done in {time.time()-t0:.0f}s  (total {(time.time()-T0)/60:.1f} min)')

R = pd.DataFrame(rows)
print('rows:', len(R), '| saved to', RES_CSV)

## 12. Paper tables (mean ± std over seeds)

In [ ]:
# ── 12. Paper tables (mean ± std over seeds) ─────────────────────────────
def pm(df, index, columns, value='macro_f1'):
    g = df.groupby(index + columns)[value]
    mean, std = g.mean().unstack(columns), g.std().fillna(0).unstack(columns)
    return mean, std, mean.round(3).astype(str) + ' ± ' + std.round(3).astype(str)

def show(title, table, fname):
    print('\n' + '=' * 90 + f'\n{title}\n' + '=' * 90)
    print(table.to_string())
    table.to_csv(os.path.join(OUT_DIR, fname + '.csv'))
    try:
        with open(os.path.join(OUT_DIR, fname + '.tex'), 'w') as fh: fh.write(table.to_latex())
    except Exception:
        pass

main = R[R.exp == 'main']
order = [m for m in METHODS]
for atk in ATTACKS:
    mean, std, txt = pm(main[main.attack == atk], ['dataset'], ['method'])
    txt = txt[order]; txt['centralised LR (no attack)'] = [f"{CENTRAL[d]['macro_f1']:.3f}" for d in txt.index]
    show(f'TABLE 1{"abc"[ATTACKS.index(atk)]} – Macro-F1, 20% Byzantine ({atk}), stealth drift', txt, f'table1_{atk}')

_, _, txt = pm(main[main.attack == 'sign_flip'], ['dataset'], ['method'], 'bal')
show('TABLE 2 – Balanced accuracy, 20% Byzantine (sign_flip), stealth drift', txt[order], 'table2_balacc')

det = main[main.method == 'ARC-Trust+'].groupby(['dataset', 'attack'])[['iso_recall', 'false_iso', 'trust_honest', 'trust_byz']].mean().round(3)
show('TABLE 3 – ARC-Trust+ detection: share of Byzantine links isolated / honest links wrongly isolated / mean trust', det, 'table3_detection')

abl = R[R.exp == 'ablation'].copy()
full = main[main.method == 'ARC-Trust+'].copy(); full['variant'] = 'Full ARC-Trust+'
abl = pd.concat([full, abl])
t4 = abl.groupby(['variant', 'attack'])[['macro_f1', 'iso_recall', 'false_iso']].mean().round(3).unstack('attack')
t4 = t4.reindex(list(ABLATIONS))
show('TABLE 4 – Ablation (mean over datasets and seeds)', t4, 'table4_ablation')

_, _, t5 = pm(R[R.exp == 'byz_frac'], ['dataset', 'method'], ['byz_frac'])
show('TABLE 5 – Macro-F1 vs Byzantine fraction (sign_flip)', t5, 'table5_byzfrac')

_, _, t6 = pm(R[R.exp == 'drift'], ['dataset', 'method'], ['drift'])
show('TABLE 6 – Macro-F1 vs drift type (sign_flip, 20% Byzantine)', t6[DRIFTS], 'table6_drift')

## 13. Figures (saved as PNG + PDF in OUT_DIR)

In [ ]:
# ── 13. Figures (saved as PNG + PDF in OUT_DIR) ──────────────────────────
COL = {'ARC-Trust+': '#1b7837', 'fedavg': '#999999', 'median': '#2166ac', 'multi_krum': '#b2182b',
       'sync': '#bbbbbb', 'distance_only': '#e08214', 'canary_only': '#8073ac'}
def savefig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, name + '.png'), bbox_inches='tight')
    fig.savefig(os.path.join(OUT_DIR, name + '.pdf'), bbox_inches='tight')
    plt.show()

def mean_curve(name, method, atk, key):
    arr = [HIST[(name, s, method, atk)][key] for s in SEEDS if (name, s, method, atk) in HIST]
    return np.nanmean(arr, 0) if arr else None

# Fig 1 – macro-F1 over rounds per dataset (sign_flip)
fig, axes = plt.subplots(1, len(DATASETS), figsize=(4 * len(DATASETS), 3.4), sharey=True)
for ax, name in zip(np.atleast_1d(axes), DATASETS):
    for m in METHODS:
        c = mean_curve(name, m, 'sign_flip', 'macro_f1')
        if c is not None:
            ax.plot(c, color=COL[m], lw=2.2 if m == 'ARC-Trust+' else 1.2, label=m)
    ax.axvline(HP['DRIFT_START'], color='k', ls=':', lw=1)
    ax.set_title(name); ax.set_xlabel('round'); ax.grid(alpha=.3)
np.atleast_1d(axes)[0].set_ylabel('macro-F1 (honest agents)')
np.atleast_1d(axes)[-1].legend(fontsize=7, loc='lower right')
fig.suptitle('Macro-F1 over rounds, 20% Byzantine (sign-flip), stealth drift from round 18', y=1.03)
savefig(fig, 'fig1_f1_over_rounds')

# Fig 2 – trust trajectories and isolation
fig, axes = plt.subplots(1, len(DATASETS), figsize=(4 * len(DATASETS), 3.2), sharey=True)
for ax, name in zip(np.atleast_1d(axes), DATASETS):
    for atk, ls in zip(ATTACKS, ['-', '--', ':']):
        ax.plot(mean_curve(name, 'ARC-Trust+', atk, 'trust_honest'), color='#2166ac', ls=ls, label=f'honest ({atk})')
        ax.plot(mean_curve(name, 'ARC-Trust+', atk, 'trust_byz'), color='#b2182b', ls=ls, label=f'Byzantine ({atk})')
    ax.axhline(HP['TAU_ISO'], color='k', lw=1, ls='-.')
    ax.set_title(name); ax.set_xlabel('round'); ax.grid(alpha=.3)
np.atleast_1d(axes)[0].set_ylabel('mean trust'); np.atleast_1d(axes)[-1].legend(fontsize=6)
fig.suptitle('ARC-Trust+ trust in honest vs Byzantine neighbours (dash-dot = isolation threshold)', y=1.03)
savefig(fig, 'fig2_trust')

# Fig 3 – main comparison bars
mean, std, _ = pm(main[main.attack == 'sign_flip'], ['dataset'], ['method'])
fig, ax = plt.subplots(figsize=(11, 3.8)); w = 0.8 / len(METHODS); x = np.arange(len(mean.index))
for k, m in enumerate(METHODS):
    ax.bar(x + k * w, mean[m], w, yerr=std[m], color=COL[m], label=m, capsize=2)
ax.set_xticks(x + 0.4 - w / 2); ax.set_xticklabels(mean.index); ax.set_ylabel('macro-F1')
ax.set_ylim(max(0, np.nanmin(mean.values) - 0.1), 1.0); ax.legend(ncol=4, fontsize=8); ax.grid(axis='y', alpha=.3)
ax.set_title('Final macro-F1 (last 10 rounds), 20% Byzantine sign-flip, mean ± std over seeds')
savefig(fig, 'fig3_main_bars')

# Fig 4 – robustness vs Byzantine fraction
bf = R[R.exp == 'byz_frac'].groupby(['dataset', 'method', 'byz_frac']).macro_f1.mean()
fig, axes = plt.subplots(1, len(DATASETS), figsize=(4 * len(DATASETS), 3.2), sharey=True)
for ax, name in zip(np.atleast_1d(axes), DATASETS):
    for m in SENS_METHODS:
        s = bf.loc[name, m]; ax.plot(np.array(s.index) * 100, s.values, 'o-', color=COL[m], label=m)
    ax.set_title(name); ax.set_xlabel('Byzantine agents (%)'); ax.grid(alpha=.3)
np.atleast_1d(axes)[0].set_ylabel('macro-F1'); np.atleast_1d(axes)[-1].legend(fontsize=7)
savefig(fig, 'fig4_byz_fraction')
print('Figures saved to', OUT_DIR)

## 14. Sanity check: ORIGINAL (legacy) logic vs corrected logic

In [ ]:
# ── 14. Sanity check: ORIGINAL (legacy) logic vs corrected logic ─────────
# Same data, seed and attack. Shows how the original protocol flattered ARC-Trust+:
# its Byzantine agents never poisoned the model, and isolation could not fire.
_prev = LEGACY
cmp_rows = []
for flag in [True, False]:
    LEGACY = flag
    st = build_streams(DATA[DATASETS[0]], 0, 'stealth')
    for m in ['ARC-Trust+', 'fedavg', 'median', 'multi_krum', 'distance_only']:
        s = summarise(run_method(m, DATA[DATASETS[0]], st, 0, 0.2, 'sign_flip'))
        cmp_rows.append(dict(logic='original' if flag else 'corrected', method=m, macro_f1=round(s['macro_f1'], 3),
                             iso_recall=round(s.get('iso_recall', np.nan), 2)))
LEGACY = _prev
print(pd.DataFrame(cmp_rows).pivot(index='method', columns='logic', values=['macro_f1', 'iso_recall']).to_string())

## 15. Download everything (tables .csv/.tex, figures .png/.pdf, raw results)

In [ ]:
# ── 15. Download everything (tables .csv/.tex, figures .png/.pdf, raw results) ──
import shutil
zip_path = shutil.make_archive(OUT_DIR.rstrip('/') + '_arc_trust', 'zip', OUT_DIR)
print('Archive:', zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

# ============================================================
# COMPLETE RESULT AUDIT — PRINT ALL RELEVANT TABLES
# Run AFTER the experiments are finished
# ============================================================

import pandas as pd
import numpy as np
import os

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 100)

print("\n" + "=" * 120)
print("ARC-TRUST+ — COMPLETE EXPERIMENTAL RESULT AUDIT")
print("=" * 120)

# ------------------------------------------------------------
# 0. DATASET / RESULT OVERVIEW
# ------------------------------------------------------------

print("\n[0] RESULT DATASET OVERVIEW")
print("-" * 120)

print("Total result rows:", len(R))
print("Experiments:", sorted(R["exp"].dropna().unique()))
print("Datasets:", sorted(R["dataset"].dropna().unique()))
print("Methods:", sorted(R["method"].dropna().unique()))

if "attack" in R.columns:
    print("Attacks:", sorted(R["attack"].dropna().unique()))

if "byz_frac" in R.columns:
    print("Byzantine fractions:",
          sorted(R["byz_frac"].dropna().unique()))

if "drift" in R.columns:
    print("Drift types:",
          sorted(R["drift"].dropna().unique()))

if "seed" in R.columns:
    print("Seeds:", sorted(R["seed"].dropna().unique()))

print("\nRows by experiment:")
print(R.groupby("exp").size().to_string())


# ------------------------------------------------------------
# 1. RAW MAIN RESULTS
# ------------------------------------------------------------

main = R[R["exp"] == "main"].copy()

print("\n" + "=" * 120)
print("[1] RAW MAIN EXPERIMENT RESULTS")
print("=" * 120)

print(main.to_string(index=False))


# ------------------------------------------------------------
# 2. MAIN RESULTS — MACRO F1
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[2] MACRO-F1 — MEAN ± STD")
print("=" * 120)

for attack in sorted(main["attack"].dropna().unique()):

    tmp = main[main["attack"] == attack]

    stats = (
        tmp.groupby(["dataset", "method"])["macro_f1"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    stats["result"] = (
        stats["mean"].round(4).astype(str)
        + " ± "
        + stats["std"].fillna(0).round(4).astype(str)
    )

    table = stats.pivot(
        index="dataset",
        columns="method",
        values="result"
    )

    print("\nATTACK:", attack)
    print(table.to_string())


# ------------------------------------------------------------
# 3. BALANCED ACCURACY
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[3] BALANCED ACCURACY — SIGN FLIP")
print("=" * 120)

tmp = main[main["attack"] == "sign_flip"]

stats = (
    tmp.groupby(["dataset", "method"])["bal"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

stats["result"] = (
    stats["mean"].round(4).astype(str)
    + " ± "
    + stats["std"].fillna(0).round(4).astype(str)
)

print(
    stats.pivot(
        index="dataset",
        columns="method",
        values="result"
    ).to_string()
)


# ------------------------------------------------------------
# 4. ACCURACY / PRECISION / RECALL / F1
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[4] ALL CLASSIFICATION METRICS — MAIN EXPERIMENT")
print("=" * 120)

metric_candidates = [
    "accuracy",
    "acc",
    "bal",
    "precision",
    "recall",
    "specificity",
    "macro_f1",
    "attack_f1",
    "weighted_f1"
]

available_metrics = [
    c for c in metric_candidates if c in main.columns
]

print("Available metrics:", available_metrics)

for metric in available_metrics:

    print("\n" + "-" * 100)
    print("METRIC:", metric)
    print("-" * 100)

    stats = (
        main.groupby(["dataset", "method"])[metric]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    stats["result"] = (
        stats["mean"].round(4).astype(str)
        + " ± "
        + stats["std"].fillna(0).round(4).astype(str)
    )

    print(
        stats.pivot(
            index="dataset",
            columns="method",
            values="result"
        ).to_string()
    )


# ------------------------------------------------------------
# 5. ARC-TRUST+ DETECTION RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[5] ARC-TRUST+ BYZANTINE DETECTION METRICS")
print("=" * 120)

detection_cols = [
    c for c in [
        "iso_recall",
        "false_iso",
        "trust_honest",
        "trust_byz"
    ]
    if c in main.columns
]

arc = main[main["method"] == "ARC-Trust+"]

if detection_cols:

    print(
        arc.groupby(["dataset", "attack"])[detection_cols]
        .agg(["mean", "std", "count"])
        .round(4)
        .to_string()
    )

else:
    print("No Byzantine detection columns found.")


# ------------------------------------------------------------
# 6. CHECK WHICH METHODS ACTUALLY HAVE ISOLATION RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[6] ISOLATION METRIC AVAILABILITY BY METHOD")
print("=" * 120)

if "iso_recall" in R.columns:

    iso_check = (
        R.groupby("method")
        .agg(
            rows=("iso_recall", "size"),
            non_null=("iso_recall", "count"),
            mean=("iso_recall", "mean"),
            min=("iso_recall", "min"),
            max=("iso_recall", "max")
        )
        .reset_index()
    )

    print(iso_check.to_string(index=False))

    print("\nIMPORTANT:")
    print(
        "If a method has non_null = 0, its isolation performance was NOT evaluated."
    )


# ------------------------------------------------------------
# 7. FALSE ISOLATION CHECK
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[7] FALSE ISOLATION AUDIT")
print("=" * 120)

if "false_iso" in R.columns:

    print(
        R.groupby("method")["false_iso"]
        .agg(["count", "mean", "std", "min", "max"])
        .round(4)
        .to_string()
    )


# ------------------------------------------------------------
# 8. TRUST SCORE AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[8] TRUST SCORE AUDIT")
print("=" * 120)

trust_cols = [
    c for c in ["trust_honest", "trust_byz"]
    if c in R.columns
]

if trust_cols:

    print(
        R.groupby(["dataset", "method"])[trust_cols]
        .agg(["mean", "std", "min", "max"])
        .round(4)
        .to_string()
    )


# ------------------------------------------------------------
# 9. ABLATION
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[9] ABLATION RESULTS")
print("=" * 120)

abl = R[R["exp"] == "ablation"].copy()

if len(abl):

    print("\nRaw ablation results:")
    print(abl.to_string(index=False))

    ab_metrics = [
        c for c in [
            "macro_f1",
            "bal",
            "iso_recall",
            "false_iso"
        ]
        if c in abl.columns
    ]

    print("\nAblation mean ± std:")

    print(
        abl.groupby(["variant", "attack"])[ab_metrics]
        .agg(["mean", "std", "count"])
        .round(4)
        .to_string()
    )

else:
    print("No ablation results found.")


# ------------------------------------------------------------
# 10. BYZANTINE FRACTION
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[10] BYZANTINE-FRACTION EXPERIMENT")
print("=" * 120)

bf = R[R["exp"] == "byz_frac"].copy()

if len(bf):

    print("\nRaw Byzantine-fraction results:")
    print(bf.to_string(index=False))

    stats = (
        bf.groupby(
            ["dataset", "method", "byz_frac"]
        )["macro_f1"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    stats["result"] = (
        stats["mean"].round(4).astype(str)
        + " ± "
        + stats["std"].fillna(0).round(4).astype(str)
    )

    print("\nMacro-F1 vs Byzantine fraction:")

    print(
        stats.pivot_table(
            index=["dataset", "method"],
            columns="byz_frac",
            values="result",
            aggfunc="first"
        ).to_string()
    )

else:
    print("No Byzantine-fraction results found.")


# ------------------------------------------------------------
# 11. DRIFT EXPERIMENT
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[11] DRIFT EXPERIMENT")
print("=" * 120)

dr = R[R["exp"] == "drift"].copy()

if len(dr):

    print("\nRaw drift results:")
    print(dr.to_string(index=False))

    stats = (
        dr.groupby(
            ["dataset", "method", "drift"]
        )["macro_f1"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    stats["result"] = (
        stats["mean"].round(4).astype(str)
        + " ± "
        + stats["std"].fillna(0).round(4).astype(str)
    )

    print("\nMacro-F1 vs drift:")

    print(
        stats.pivot_table(
            index=["dataset", "method"],
            columns="drift",
            values="result",
            aggfunc="first"
        ).to_string()
    )

else:
    print("No drift results found.")


# ------------------------------------------------------------
# 12. CENTRALIZED REFERENCE
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[12] CENTRALIZED REFERENCE RESULTS")
print("=" * 120)

if "CENTRAL" in globals():

    central_rows = []

    for dataset, vals in CENTRAL.items():

        row = {"dataset": dataset}

        for key, value in vals.items():

            if isinstance(value, (int, float, np.number)):
                row[key] = value

        central_rows.append(row)

    central_df = pd.DataFrame(central_rows)

    print(central_df.round(4).to_string(index=False))

else:
    print("CENTRAL dictionary not found.")


# ------------------------------------------------------------
# 13. SANITY CHECK — ORIGINAL VS CORRECTED
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[13] ORIGINAL VS CORRECTED LOGIC SANITY CHECK")
print("=" * 120)

_prev = LEGACY

cmp_rows = []

for flag in [True, False]:

    LEGACY = flag

    st = build_streams(
        DATA[DATASETS[0]],
        0,
        "stealth"
    )

    for method in [
        "ARC-Trust+",
        "fedavg",
        "median",
        "multi_krum",
        "distance_only"
    ]:

        s = summarise(
            run_method(
                method,
                DATA[DATASETS[0]],
                st,
                0,
                0.2,
                "sign_flip"
            )
        )

        cmp_rows.append({
            "logic": "original" if flag else "corrected",
            "method": method,
            "macro_f1": s.get("macro_f1", np.nan),
            "balanced_accuracy": s.get("bal", np.nan),
            "iso_recall": s.get("iso_recall", np.nan),
            "false_iso": s.get("false_iso", np.nan),
            "trust_honest": s.get("trust_honest", np.nan),
            "trust_byz": s.get("trust_byz", np.nan)
        })

LEGACY = _prev

sanity_df = pd.DataFrame(cmp_rows)

print(sanity_df.round(4).to_string(index=False))


# ------------------------------------------------------------
# 14. SANITY CHECK — PIVOT
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[14] SANITY CHECK — ORIGINAL vs CORRECTED PIVOT")
print("=" * 120)

for metric in [
    "macro_f1",
    "balanced_accuracy",
    "iso_recall",
    "false_iso",
    "trust_honest",
    "trust_byz"
]:

    if metric in sanity_df.columns:

        print("\nMETRIC:", metric)

        print(
            sanity_df.pivot(
                index="method",
                columns="logic",
                values=metric
            ).round(4).to_string()
        )


# ------------------------------------------------------------
# 15. MISSINGNESS AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[15] RESULT COLUMN MISSINGNESS AUDIT")
print("=" * 120)

missing = (
    R.isna()
    .sum()
    .sort_values(ascending=False)
)

print(missing.to_string())


# ------------------------------------------------------------
# 16. DUPLICATE / REPEATED RESULT AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[16] DUPLICATE RESULT AUDIT")
print("=" * 120)

possible_keys = [
    c for c in [
        "exp",
        "dataset",
        "seed",
        "method",
        "attack",
        "byz_frac",
        "drift",
        "variant"
    ]
    if c in R.columns
]

if possible_keys:

    dup = (
        R.groupby(possible_keys)
        .size()
        .reset_index(name="count")
    )

    duplicates = dup[dup["count"] > 1]

    if len(duplicates):

        print("DUPLICATES FOUND:")
        print(duplicates.to_string(index=False))

    else:

        print("No duplicate experiment keys found.")

    print("\nExperiment key:")
    print(possible_keys)


# ------------------------------------------------------------
# 17. SEED COUNT AUDIT
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[17] SEED / REPLICATION AUDIT")
print("=" * 120)

if "seed" in R.columns:

    print(
        R.groupby(
            ["exp", "dataset", "method"]
        )["seed"]
        .nunique()
        .reset_index(name="n_seeds")
        .to_string(index=False)
    )


# ------------------------------------------------------------
# 18. MAIN RESULT RANKING — DESCRIPTIVE ONLY
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("[18] MACRO-F1 VALUES — DESCRIPTIVE COMPARISON")
print("=" * 120)

ranking = (
    main.groupby(["attack", "dataset", "method"])["macro_f1"]
    .mean()
    .reset_index()
)

for attack in sorted(ranking["attack"].unique()):

    print("\nATTACK:", attack)

    x = ranking[
        ranking["attack"] == attack
    ]

    for dataset in sorted(x["dataset"].unique()):

        y = (
            x[x["dataset"] == dataset]
            .sort_values("macro_f1", ascending=False)
        )

        print("\n", dataset)
        print(y.to_string(index=False))


# ------------------------------------------------------------
# 19. SAVE COMPLETE AUDIT TABLES
# ------------------------------------------------------------

AUDIT_DIR = os.path.join(OUT_DIR, "complete_audit")
os.makedirs(AUDIT_DIR, exist_ok=True)

R.to_csv(
    os.path.join(AUDIT_DIR, "ALL_RAW_RESULTS.csv"),
    index=False
)

main.to_csv(
    os.path.join(AUDIT_DIR, "MAIN_RESULTS_RAW.csv"),
    index=False
)

if len(abl):
    abl.to_csv(
        os.path.join(AUDIT_DIR, "ABLATION_RAW.csv"),
        index=False
    )

if len(bf):
    bf.to_csv(
        os.path.join(AUDIT_DIR, "BYZANTINE_FRACTION_RAW.csv"),
        index=False
    )

if len(dr):
    dr.to_csv(
        os.path.join(AUDIT_DIR, "DRIFT_RAW.csv"),
        index=False
    )

sanity_df.to_csv(
    os.path.join(AUDIT_DIR, "ORIGINAL_VS_CORRECTED.csv"),
    index=False
)

print("\n" + "=" * 120)
print("AUDIT COMPLETE")
print("=" * 120)
print("Saved to:", AUDIT_DIR)